# Predictive Analysis

**Authors:** Abderrahmane Salmi, Ricardo Talarico, Lorenzo Allegrini

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV 
from sklearn.pipeline import Pipeline

Generating the ground truth labels

In [15]:
tracks_df = pd.read_csv("datasets/tracks_cleaned.csv")
artists_df = pd.read_csv("datasets/artists_cleaned.csv")

In [16]:
region_to_class = {
    "Lombardia": "Lombardia",
    "Piemonte": "North",
    "Trentino": "North",
    "Liguria": "North",
    "Veneto":"North",
    "Friuli-Venezia-Giulia": "North",
    "Valle d'Aosta": "North",
    "Emilia-Romagna": "Center",
    "Toscana":"Center",
    "Abruzzo":"Center",
    "Umbria":"Lazio",
    "Lazio":"Lazio",
    "Marche":"Lazio",
    "Molise":"Campania",
    "Campania":"Campania",
    "Sardegna":"Sardegna",
    "Basilicata": "South",
    "Puglia": "South",
    "Calabria":"South",
    "Sicilia":"South"
}

In [23]:
# Get all artists name to lower case
artists_df['name_norm'] = artists_df['name'].str.lower()
tracks_df['name_norm'] = tracks_df['name_artist'].str.lower()

In [ ]:
artist_to_class = (
    artists_df
    .assign(class_label=lambda df: df['region'].map(region_to_class))
    .set_index('name_norm')['class_label']
    .to_dict()
)
print(artist_to_class)


{'99 posse': nan, 'achille lauro': 'North', 'alfa': nan, 'anna pepe': nan, 'articolo 31': nan, 'babaman': 'Lombardia', 'baby k': nan, 'bassi maestro': 'Lombardia', 'beba': nan, 'bigmama': nan, 'brusco': nan, 'bushwaka': nan, 'caneda': nan, 'caparezza': 'South', 'capo plaza': 'Campania', 'chadia rodriguez': 'North', 'clementino': 'Campania', 'club dogo': nan, 'coez': 'Campania', 'colle der fomento': nan, 'cor veleno': nan, 'dani faiv': 'North', 'dargen d_amico': nan, 'dark polo gang': nan, 'doll kill': nan, 'don joe': 'Lombardia', 'drefgold': 'Center', 'emis killa': 'Lombardia', 'ensi': 'North', 'entics': 'Lombardia', 'ernia': 'Lombardia', 'eva rea': nan, 'fabri fibra': nan, 'fedez': 'Lombardia', 'frah quintale': 'Lombardia', 'frankie hi-nrg mc': 'North', 'fred de palma': 'North', 'gemitaiz': 'Lazio', 'geolier': 'Campania', 'ghali': 'Lombardia', 'ghemon': 'Campania', 'grido': 'Lombardia', 'guè pequeno': nan, 'hell raton': 'Sardegna', 'hindaco': nan, 'il tre': 'Lazio', 'inoki': 'Lazio', 

In [25]:
tracks_df['class_label'] = tracks_df['name_norm'].map(artist_to_class)
tracks_df['class_label'].unique()

array(['North', nan, 'Lazio', 'Campania', 'Lombardia', 'Center', 'South',
       'Sardegna'], dtype=object)

In [20]:
len(tracks_df['name_artist'].unique())

104

In [14]:
# Defining features useful for classification
classification_features = [
    "swear_IT","swear_EN","year","n_tokens","tokens_per_sent","char_per_tok",
    "lexical_density","avg_token_per_clause","bpm","centroid","rolloff","flux","flatness",
    "spectral_complexity","pitch","loudness","album_type","explicit","popularity","duration_sec",
    "swear_ratio","aggressiveness","relative_popularity","release_season","has_collaboration","song_age"
    ]

In [10]:
# Defining a seed for experiments
seed = 42
# Defining the number of fold for the k-fold validation
n_folds = 5

In [ ]:
# Separating ground-truth labels from features
y_true = tracks_df['class_label']
X = tracks_df.drop(columns=['class_label'])
X = tracks_df[[classification_features]]
X.head()

In [11]:
# Scaling the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

NameError: name 'X' is not defined

In [12]:
# Defining the models and their hyperparameter grids
pipe = Pipeline([('model',RidgeClassifier())])
grids = [
    {
        'model':[DecisionTreeClassifier()],
        'model__criterion': ['gini', 'entropy'],
        'model__max_depth': [None, 5, 10, 20, 40],
        'model__min_samples_split': [2, 5, 10, 20],
        'model__min_samples_leaf': [1, 2, 4, 8],
        'model__max_features': [None, 'sqrt', 'log2'],
        'model__class_weight': [None, 'balanced']
    },
    {
        'model':[KNeighborsClassifier()],
        'model__n_neighbors': [3, 5, 7, 11, 15],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean','minkowski'],
        'model__p': [1, 2]      # only used for minkowski
    },
    {
        'model':[RidgeClassifier()],
        'model__alpha': [0.1, 1.0, 10.0, 100.0],
        'model__fit_intercept': [True, False],
        'model__tol': [1e-3, 1e-4],
        'model__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga']
    }

]

In [13]:
# Defining the validation procedure
stratKFold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
gridsearch = GridSearchCV(
    estimator=pipe,
    param_grid=grids,
    cv = stratKFold,
    n_jobs=4
    )